# Model Stability Audit: Repeated Customer-Grouped Splits

This notebook measures how stable the results of the six experiment cells are (three feature sets, two releases) when the train and test partitions change.

Each cell was evaluated on one customer-grouped split (seed 42) whose test partition has only 59 rows, so one test row moves a metric by about 1.7 percentage points. A single split cannot show how much of a difference between two cells is ordinary split-to-split variation. Here the partitioning is repeated with 50 different seeds, the model is refitted on every split, and the variation of each metric is reported.

## Design

- **Splits.** For each of the 50 seeds (1 to 51, without 42, the seed of the modelling notebooks) the table is partitioned as in the modelling notebooks: two successive grouped shuffle splits by `Customer_ID` (70% train, then the remainder halved into validation and test), using the source customer labels. The same partitions are applied to both releases through `Property_ID`.
- **Model.** One model, with no tuning and no selection: the Random Forest of notebook 03 (500 trees, balanced subsample class weights, seed 42, all other settings at their defaults). Its predicted class gives Top-1 accuracy, Macro F1 and Weighted F1, and its ranked class probabilities give Top-2 and Top-3 accuracy. It is refitted on the training partition of every split.
- **Final configuration of the ranking forest.** The ranking forest saved by the modelling notebooks, whose setting was selected by a search, is also refitted on every split without re-tuning. For each feature set the setting of the source cell is used for both releases, so that the paired comparison between releases does not mix different settings. Its Top-2 and Top-3 are reported next to those of the default forest.
- **Reference.** A class-prior baseline ranks the classes by their frequency in the training partition.
- **Paired comparisons.** Differences are computed within each split: anonymised release minus source release for each feature set, the full feature set minus the no-AI feature set (with AI minus without AI) and the pure-record feature set minus the full feature set within each release.
- **What is reported.** Mean, standard deviation and the 2.5th and 97.5th percentiles over the 50 splits. The values are descriptive. The splits are drawn from the same 431 states and overlap, so the spread is not the sampling variability of a new set of customers, and no significance statement is made. The test partitions differ in size from split to split (about 53 to 83 rows). The settings of the saved ranking forests were selected by searches on the seed-42 split, so Top-2 and Top-3 obtained with them can be somewhat optimistic; the default forest involves no selection.
- **Setting sensitivity.** The setting of the ranking forest is not chosen. Its influence on Top-2 and Top-3 is shown in a separate section by refitting it with every combination of the values selected by the six searches of the modelling notebooks (12 settings).
- **Where the results are kept.** Every table is printed in the output of the cell that computes it, and the interpretation at the end of this notebook summarises them. No result files are written.
- **Difference from the modelling notebooks.** The modelling notebooks use a regularised Random Forest, selected by a search, for Top-2 and Top-3. Here the default forest gives all main metrics, so that no setting has to be chosen, and the final configuration of the ranking forest is added as a second source of Top-2 and Top-3. The Top-2 and Top-3 of the default forest are therefore not comparable with the saved single-split values; Top-1 accuracy, Macro F1 and Weighted F1 are.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

DATA_DIR = Path("../../data/processed/hypothesis_testing")

SEEDS = [seed for seed in range(1, 52) if seed != 42]
CHECK_SEED = 42

## Data

Both modelling tables have the same rows in the same order. The check below confirms that the keys, timestamps and targets are identical in the two releases, so the same row indices can be used for both.

In [ ]:
source = pd.read_csv(DATA_DIR / "source_dataset" / "model_state.csv", parse_dates=["Prediction_Time"])
regions = pd.read_csv(DATA_DIR / "regions_anonymised" / "model_state.csv", parse_dates=["Prediction_Time"])

for column in ["Property_ID", "Prediction_Time", "Next_Category"]:
    assert (source[column].values == regions[column].values).all(), column

RELEASES = {"source": source, "regions": regions}

print("DATA LOADED")
print("-" * 60)
print(f"Rows:       {len(source):,}")
print(f"Properties: {source['Property_ID'].nunique():,}")
print(f"Customers:  {source['Customer_ID'].nunique():,}")

## Feature Sets And Model

The three feature sets are the ones used in the modelling notebooks: all 55 predictors, the 42 predictors without the `ai_` columns, and the 12 documented-record predictors. The model is the Random Forest of notebook 03.

In [ ]:
IDENTIFIERS = ["Property_ID", "Customer_ID", "Prediction_Time", "Next_Category_ID", "Next_Category"]

ALL_FEATURES = [column for column in source.columns if column not in IDENTIFIERS]
NO_AI_FEATURES = [column for column in ALL_FEATURES if not column.startswith("ai_")]
RECORD_FEATURES = [
    "documented_asset_count",
    "documented_category_count",
    "documented_value_total",
    "documented_replacement_cost_total",
    "mean_evidence_rating",
    "documented_count_CAT-001",
    "documented_count_CAT-002",
    "documented_count_CAT-003",
    "documented_count_CAT-004",
    "documented_count_CAT-005",
    "documented_count_CAT-006",
    "documented_count_CAT-007",
]

FEATURE_SETS = {
    "full": ALL_FEATURES,
    "no_ai": NO_AI_FEATURES,
    "pure_record": RECORD_FEATURES,
}

assert [len(columns) for columns in FEATURE_SETS.values()] == [55, 42, 12]

# The Random Forest of notebook 03: no search, no tuning.
FOREST_SETTINGS = dict(
    n_estimators=500,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)

print("FEATURE SETS")
print("-" * 60)
for name, columns in FEATURE_SETS.items():
    print(f"{name:<12} {len(columns)} predictors")

## Check Against The Saved Models

The feature sets and the forest above are written out in this notebook. The cell below loads the six saved Top-1 models of the modelling notebooks and confirms that each one was trained on exactly these predictors and with exactly these forest settings, so that nothing in this notebook rests on an unchecked assumption.

In [ ]:
import joblib

MODELS_DIR = Path("../../models/hypothesis_testing")

print("CHECK AGAINST THE SAVED TOP-1 MODELS")
print("-" * 80)

for release in RELEASES:
    for feature_set, columns in FEATURE_SETS.items():
        saved = joblib.load(MODELS_DIR / f"{release}_{feature_set}_top1.joblib")

        saved_columns = [
            column
            for name, _, group in saved.named_steps["preprocessor"].transformers_
            if name in ("numerical", "categorical")
            for column in group
        ]

        saved_forest = saved.named_steps["classifier"].get_params()

        assert set(saved_columns) == set(columns), (release, feature_set)
        assert len(saved_columns) == len(columns), (release, feature_set)

        for name, value in FOREST_SETTINGS.items():
            assert saved_forest[name] == value, (release, feature_set, name)

        print(
            f"{release:<8} {feature_set:<12} predictors {len(saved_columns):>2} match | "
            f"forest settings match ({saved_forest['n_estimators']} trees)"
        )

print()
print("The saved Top-1 models use exactly the predictors and settings defined above.")

## Functions

The preprocessing is the one of the modelling notebooks (median imputation and scaling for numeric columns, mode imputation and one-hot encoding for the others) and is fitted on the training partition only. Top-k accuracy is computed from the ranked class probabilities; a true class that is absent from a training partition counts as a miss.

In [ ]:
def build_pipeline(frame, columns):
    numeric = frame[columns].select_dtypes(include=["number"]).columns.tolist()
    categorical = [column for column in columns if column not in numeric]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "numerical",
                Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]),
                numeric,
            ),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("encoder", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical,
            ),
        ]
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(**FOREST_SETTINGS)),
    ])


def top_k_accuracy(model, X, y, k):
    probabilities = model.predict_proba(X)
    classes = model.named_steps["classifier"].classes_
    ranked = classes[np.argsort(-probabilities, axis=1, kind="stable")[:, :k]]
    return float(np.mean([truth in row for truth, row in zip(y, ranked)]))


def customer_grouped_split(seed):
    groups = source["Customer_ID"]
    target = source["Next_Category"]

    train_idx, temp_idx = next(
        GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=seed).split(
            source, target, groups=groups
        )
    )

    val_rel, test_rel = next(
        GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed).split(
            source.iloc[temp_idx], target.iloc[temp_idx], groups=groups.iloc[temp_idx]
        )
    )

    return train_idx, temp_idx[val_rel], temp_idx[test_rel]


def class_prior_baseline(y_train, y_test):
    counts = y_train.value_counts()
    order = sorted(counts.index, key=lambda label: (-counts[label], label))

    predictions = np.repeat(order[0], len(y_test))

    return dict(
        top1_accuracy=accuracy_score(y_test, predictions),
        macro_f1=f1_score(y_test, predictions, average="macro", zero_division=0),
        weighted_f1=f1_score(y_test, predictions, average="weighted", zero_division=0),
        top2_accuracy=float(np.mean([truth in order[:2] for truth in y_test])),
        top3_accuracy=float(np.mean([truth in order[:3] for truth in y_test])),
    )


def ranking_pipeline(frame, columns, settings):
    pipeline = build_pipeline(frame, columns)
    pipeline.set_params(**{f"classifier__{name}": value for name, value in settings.items()})
    return pipeline


def evaluate(release, feature_set, train_idx, test_idx, with_final=False):
    frame = RELEASES[release]
    columns = FEATURE_SETS[feature_set]
    X_train, X_test = frame[columns].iloc[train_idx], frame[columns].iloc[test_idx]
    y_train, y_test = frame["Next_Category"].iloc[train_idx], frame["Next_Category"].iloc[test_idx]

    model = build_pipeline(frame, columns).fit(X_train, y_train)
    predictions = model.predict(X_test)

    result = dict(
        top1_accuracy=accuracy_score(y_test, predictions),
        macro_f1=f1_score(y_test, predictions, average="macro", zero_division=0),
        weighted_f1=f1_score(y_test, predictions, average="weighted", zero_division=0),
        top2_accuracy=top_k_accuracy(model, X_test, y_test.values, 2),
        top3_accuracy=top_k_accuracy(model, X_test, y_test.values, 3),
    )

    if with_final:
        ranking = ranking_pipeline(frame, columns, FINAL_SETTINGS[feature_set]).fit(X_train, y_train)
        result["final_top2_accuracy"] = top_k_accuracy(ranking, X_test, y_test.values, 2)
        result["final_top3_accuracy"] = top_k_accuracy(ranking, X_test, y_test.values, 3)

    return result

## Reproduction Check

Before the repetitions, the procedure is run on the seed-42 split used in the modelling notebooks. The Top-1 metrics of every cell must equal the holdout results saved by the corresponding notebook, which confirms that the splits, preprocessing and Top-1 model are the same.

In [ ]:
FEATURE_FOLDERS = {
    "full": "01_full_feature_set",
    "no_ai": "02_no_ai_features",
    "pure_record": "03_pure_record",
}
RELEASE_FOLDERS = {"source": "source_dataset", "regions": "regions_anonymised"}

train_idx, val_idx, test_idx = customer_grouped_split(CHECK_SEED)

print("REPRODUCTION OF THE SEED-42 TOP-1 RESULTS")
print("-" * 80)

for release, release_folder in RELEASE_FOLDERS.items():
    for feature_set, feature_folder in FEATURE_FOLDERS.items():
        saved = pd.read_csv(
            DATA_DIR / release_folder / feature_folder / "results_holdout_test.csv"
        ).iloc[0]

        result = evaluate(release, feature_set, train_idx, test_idx)

        differences = {
            "top1_accuracy": abs(result["top1_accuracy"] - saved["Top1_Accuracy"]),
            "macro_f1": abs(result["macro_f1"] - saved["Top1_Macro_F1"]),
            "weighted_f1": abs(result["weighted_f1"] - saved["Top1_Weighted_F1"]),
        }

        print(
            f"{release:<8} {feature_set:<12} "
            f"accuracy {result['top1_accuracy']:.4f} | Macro F1 {result['macro_f1']:.4f} | "
            f"max difference to saved {max(differences.values()):.2e}"
        )

        assert max(differences.values()) < 1e-9, (release, feature_set)

print()
print("All Top-1 metrics reproduce the saved holdout results.")

## Final Configuration Of The Ranking Forest

The modelling notebooks selected the setting of the ranking forest by a random search in each of the six cells and saved the selected forest. The cell below loads the six saved ranking forests and reads the setting that each one actually uses, so that no setting is typed by hand. The searches chose different settings in different cells and reached similar cross-validation Macro F1 scores (0.39 to 0.44, as reported in the modelling notebooks), so no setting stood out.

For the repeated splits, the final configuration of each feature set is the setting of its **source** cell, used for both releases.

In [ ]:
import joblib

MODELS_DIR = Path("../../models/hypothesis_testing")

PARAMETERS = ["n_estimators", "min_samples_split", "min_samples_leaf", "max_features", "max_depth"]

selected_rows = []

for release in RELEASE_FOLDERS:
    for feature_set in FEATURE_FOLDERS:
        saved_forest = joblib.load(MODELS_DIR / f"{release}_{feature_set}_ranking.joblib")
        forest = saved_forest.named_steps["classifier"].get_params()
        selected_rows.append(dict(release=release, feature_set=feature_set, **{name: forest[name] for name in PARAMETERS}))

print("SETTINGS OF THE SAVED RANKING FORESTS (read from the saved models)")
print("-" * 100)
print(pd.DataFrame(selected_rows).to_string(index=False))

FINAL_SETTINGS = {
    row["feature_set"]: {name: row[name] for name in PARAMETERS}
    for row in selected_rows
    if row["release"] == "source"
}

print()
print("FINAL CONFIGURATION USED FOR BOTH RELEASES (setting of the source cell)")
print("-" * 100)
for feature_set, settings in FINAL_SETTINGS.items():
    print(f"{feature_set:<12} {settings}")

## Repeated Splits

For each of the 50 seeds the model is refitted on the training partition and scored on the test partition, for both releases and all three feature sets. Nothing is tuned or selected on any split.

In [ ]:
records = []
baselines = []

for seed in SEEDS:
    train_idx, val_idx, test_idx = customer_grouped_split(seed)

    y_train = source["Next_Category"].iloc[train_idx]
    y_test = source["Next_Category"].iloc[test_idx]

    baselines.append(dict(seed=seed, **class_prior_baseline(y_train, y_test)))

    for release in RELEASES:
        for feature_set in FEATURE_SETS:
            records.append(
                dict(
                    seed=seed,
                    release=release,
                    feature_set=feature_set,
                    test_rows=len(test_idx),
                    test_customers=int(source["Customer_ID"].iloc[test_idx].nunique()),
                    **evaluate(release, feature_set, train_idx, test_idx, with_final=True),
                )
            )

per_split = pd.DataFrame(records)
baseline_per_split = pd.DataFrame(baselines)


print("REPEATED SPLITS COMPLETE")
print("-" * 60)
print(f"Splits:               {len(SEEDS)}")
print(f"Result rows:          {len(per_split)}")
print(
    f"Test rows per split:  min {per_split['test_rows'].min()} | "
    f"mean {per_split['test_rows'].mean():.1f} | max {per_split['test_rows'].max()}"
)

## Spread Of Each Cell

Mean, standard deviation and the 2.5th and 97.5th percentiles of every metric over the 50 splits. The class-prior baseline is the same for both releases because it uses only the training labels.

In [ ]:
METRICS = {
    "top1_accuracy": "Top-1 accuracy",
    "macro_f1": "Macro F1",
    "weighted_f1": "Weighted F1",
    "top2_accuracy": "Top-2 accuracy",
    "top3_accuracy": "Top-3 accuracy",
}

FINAL_METRICS = {
    "final_top2_accuracy": "Top-2 accuracy (final configuration)",
    "final_top3_accuracy": "Top-3 accuracy (final configuration)",
}

ALL_METRICS = {**METRICS, **FINAL_METRICS}


def describe(values):
    return pd.Series(
        {
            "mean": values.mean(),
            "sd": values.std(ddof=1),
            "p2.5": values.quantile(0.025),
            "p97.5": values.quantile(0.975),
        }
    )


spread_rows = []

for (release, feature_set), group in per_split.groupby(["release", "feature_set"]):
    for column, label in ALL_METRICS.items():
        spread_rows.append(
            dict(release=release, feature_set=feature_set, metric=label, **describe(group[column]))
        )

for column, label in METRICS.items():
    spread_rows.append(
        dict(release="both", feature_set="class_prior_baseline", metric=label, **describe(baseline_per_split[column]))
    )

spread = pd.DataFrame(spread_rows)

print("SPREAD OVER 50 SPLITS")
print("-" * 100)

for (release, feature_set), group in spread.groupby(["release", "feature_set"], sort=False):
    print(f"\n{release} / {feature_set}")
    print(group.drop(columns=["release", "feature_set"]).round(4).to_string(index=False))

## Paired Differences Between The Releases

Within each split, the anonymised release minus the source release, for each feature set. A positive value means that the anonymised release scored higher. The last columns give the share of splits with a negative, zero and positive difference.

In [ ]:
def paired_differences(a, b, comparison):
    rows = []
    for column, label in ALL_METRICS.items():
        difference = (a.set_index("seed")[column] - b.set_index("seed")[column]).dropna()
        rows.append(
            dict(
                comparison=comparison,
                metric=label,
                mean_difference=difference.mean(),
                sd=difference.std(ddof=1),
                **{"p2.5": difference.quantile(0.025), "p97.5": difference.quantile(0.975)},
                share_negative=float((difference < -1e-12).mean()),
                share_zero=float((difference.abs() <= 1e-12).mean()),
                share_positive=float((difference > 1e-12).mean()),
            )
        )
    return rows


def select(release, feature_set):
    return per_split[(per_split.release == release) & (per_split.feature_set == feature_set)]


release_differences = []
for feature_set in FEATURE_SETS:
    release_differences += paired_differences(
        select("regions", feature_set),
        select("source", feature_set),
        f"regions minus source | {feature_set}",
    )

release_differences = pd.DataFrame(release_differences)

print("PAIRED DIFFERENCES: REGIONS MINUS SOURCE")
print("-" * 110)

for comparison, group in release_differences.groupby("comparison", sort=False):
    print(f"\n{comparison}")
    print(group.drop(columns="comparison").round(4).to_string(index=False))

## Paired Differences Between Feature Sets

Within each release and each split: the full feature set minus the no-AI feature set (with AI minus without AI), and the pure-record feature set minus the full feature set.

In [ ]:
feature_set_differences = []

for release in RELEASES:
    full = select(release, "full")
    feature_set_differences += paired_differences(full, select(release, "no_ai"), f"{release} | with AI minus without AI (full minus no-AI)")
    feature_set_differences += paired_differences(select(release, "pure_record"), full, f"{release} | pure record minus full")

feature_set_differences = pd.DataFrame(feature_set_differences)

print("PAIRED DIFFERENCES: FEATURE SETS WITHIN A RELEASE")
print("-" * 110)

for comparison, group in feature_set_differences.groupby("comparison", sort=False):
    print(f"\n{comparison}")
    print(group.drop(columns="comparison").round(4).to_string(index=False))

## Sensitivity To The Ranking-Forest Setting

The modelling notebooks selected the setting of the ranking forest by a random search in each of the six cells and saved the selected forest. The main analysis uses the default forest of notebook 03 for the main metrics, and the final configuration of the ranking forest is reported next to it. To show how much Top-2 and Top-3 depend on the setting, the ranking forest is refitted on the same 50 splits with **every combination of the values that appear in the saved ranking forests** (table above). For each cell the spread of the mean Top-2 and Top-3 across these settings is compared with the split-to-split standard deviation of the default forest. No setting is selected.

In [ ]:
from itertools import product

OBSERVED_VALUES = {
    name: sorted({row[name] for row in selected_rows}, key=lambda value: (value is None, value or 0))
    for name in PARAMETERS
}

RANKING_SETTINGS = [
    dict(zip(PARAMETERS, values))
    for values in product(*OBSERVED_VALUES.values())
]

print("VALUES THAT APPEAR IN THE SAVED FORESTS")
print("-" * 60)
for name, values in OBSERVED_VALUES.items():
    print(f"{name:<20} {values}")

print()
print(f"Combinations: {len(RANKING_SETTINGS)}")

assert len(RANKING_SETTINGS) == 12


def setting_label(settings):
    depth = "none" if settings["max_depth"] is None else settings["max_depth"]
    return f"split {settings['min_samples_split']} / leaf {settings['min_samples_leaf']} / depth {depth}"


sensitivity_rows = []

for seed in SEEDS:
    train_idx, val_idx, test_idx = customer_grouped_split(seed)

    for release, frame in RELEASES.items():
        for feature_set, columns in FEATURE_SETS.items():
            X_train, X_test = frame[columns].iloc[train_idx], frame[columns].iloc[test_idx]
            y_train, y_test = frame["Next_Category"].iloc[train_idx], frame["Next_Category"].iloc[test_idx]

            for settings in RANKING_SETTINGS:
                model = ranking_pipeline(frame, columns, settings).fit(X_train, y_train)

                sensitivity_rows.append(
                    dict(
                        seed=seed,
                        release=release,
                        feature_set=feature_set,
                        setting=setting_label(settings),
                        top2_accuracy=top_k_accuracy(model, X_test, y_test.values, 2),
                        top3_accuracy=top_k_accuracy(model, X_test, y_test.values, 3),
                    )
                )

sensitivity = pd.DataFrame(sensitivity_rows)

print()
print("RANKING-FOREST SETTINGS EVALUATED")
print("-" * 60)
print(f"Settings:     {len(RANKING_SETTINGS)}")
print(f"Splits:       {len(SEEDS)}")
print(f"Result rows:  {len(sensitivity)}")

In [ ]:
settings_mean = (
    sensitivity
    .groupby(["release", "feature_set", "setting"])[["top2_accuracy", "top3_accuracy"]]
    .mean()
)

summary_rows = []

for (release, feature_set), group in settings_mean.groupby(level=[0, 1]):
    default = per_split[(per_split.release == release) & (per_split.feature_set == feature_set)]

    for column, label in [("top2_accuracy", "Top-2 accuracy"), ("top3_accuracy", "Top-3 accuracy")]:
        values = group[column]
        summary_rows.append(
            dict(
                release=release,
                feature_set=feature_set,
                metric=label,
                lowest_setting_mean=values.min(),
                highest_setting_mean=values.max(),
                range_across_settings=values.max() - values.min(),
                default_forest_mean=default[column].mean(),
                split_to_split_sd=default[column].std(ddof=1),
            )
        )

sensitivity_summary = pd.DataFrame(summary_rows)

print("SPREAD ACROSS THE 12 SETTINGS COMPARED WITH THE SPLIT-TO-SPLIT SPREAD")
print("-" * 110)
print(sensitivity_summary.round(4).to_string(index=False))

## Results And Interpretation

(Written after the run; see the cells above for the numbers.)